In [ ]:
# Copyright (c) Meta Platforms, Inc. and affiliates.

## 1. Imports and Model Loading

In [ ]:
import os
import sys

# 设置 CUDA_HOME 环境变量（gsplat 需要这个来检测 CUDA toolkit）
# 必须在导入任何使用 gsplat 的模块之前设置
os.environ["CUDA_HOME"] = os.environ.get("CONDA_PREFIX", "")
os.environ["LIDRA_SKIP_INIT"] = "true"

# 添加项目根目录到 Python 路径，以便导入 sam3d_objects
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

import imageio
import uuid
from IPython.display import Image as ImageDisplay

# 验证 gsplat CUDA 支持
try:
    import gsplat
    import gsplat.cuda._backend
    _C = getattr(gsplat.cuda._backend, '_C', None)
    if _C is not None:
        print("✓ gsplat CUDA 扩展已正确加载")
    else:
        print("⚠ 警告: gsplat CUDA 扩展未加载，渲染可能失败")
except Exception as e:
    print(f"⚠ gsplat 导入警告: {e}")

from inference import Inference, ready_gaussian_for_video_rendering, render_video, load_image, load_single_mask, display_image, make_scene, interactive_visualizer

In [ ]:
PATH = os.getcwd()
TAG = "hf"
config_path = f"{PATH}/../checkpoints/{TAG}/pipeline.yaml"
inference = Inference(config_path, compile=False)

## 2. Load input image to lift to 3D (single object)

In [ ]:
IMAGE_PATH = f"{PATH}/images/shutterstock_stylish_kidsroom_1640806567/image.png"
IMAGE_NAME = os.path.basename(os.path.dirname(IMAGE_PATH))

image = load_image(IMAGE_PATH)
mask = load_single_mask(os.path.dirname(IMAGE_PATH), index=14)
display_image(image, masks=[mask])

## 3. Generate Gaussian Splat

In [ ]:
# run model
output = inference(image, mask, seed=42)

# 查看 output 字典的所有键
import torch  # 导入 torch 用于类型检查

print("=" * 60)
print("output 字典包含的键:")
print("=" * 60)
for key in sorted(output.keys()):
    value = output[key]
    if isinstance(value, torch.Tensor):
        print(f"  {key:20s}: Tensor {list(value.shape)}")
    elif isinstance(value, list):
        print(f"  {key:20s}: List[{len(value)}] - {type(value[0]).__name__ if value else 'empty'}")
    elif hasattr(value, '__class__'):
        print(f"  {key:20s}: {type(value).__name__}")
    else:
        print(f"  {key:20s}: {type(value).__name__} = {value}")
print("=" * 60)

# export gaussian splat (as point cloud)
output["gs"].save_ply(f"{PATH}/gaussians/single/{IMAGE_NAME}.ply")

# export GLB file (if available)
if "glb" in output and output["glb"] is not None:
    glb_path = f"{PATH}/gaussians/single/{IMAGE_NAME}.glb"
    output["glb"].export(glb_path)
    print(f"✓ 已保存 GLB 文件: {glb_path}")
else:
    print("⚠ GLB 文件不可用（可能需要设置 decode_formats 包含 'mesh'）")

## 4. Visualize Gaussian Splat
### a. Animated Gif

In [ ]:
# render gaussian splat
scene_gs = make_scene(output)
scene_gs = ready_gaussian_for_video_rendering(scene_gs)

video = render_video(
    scene_gs,
    r=1,
    fov=60,
    pitch_deg=15,
    yaw_start_deg=-45,
    resolution=512,
)["color"]

# save video as gif
imageio.mimsave(
    os.path.join(f"{PATH}/gaussians/single/{IMAGE_NAME}.gif"),
    video,
    format="GIF",
    duration=1000 / 30,  # default assuming 30fps from the input MP4
    loop=0,  # 0 means loop indefinitely
)

# notebook display
ImageDisplay(url=f"gaussians/single/{IMAGE_NAME}.gif?cache_invalidator={uuid.uuid4()}")

### b. Interactive Visualizer

In [ ]:
# might take a while to load (black screen)
interactive_visualizer(f"{PATH}/gaussians/single/{IMAGE_NAME}.ply")